In [ ]:
import pandas as pd
import psycopg2
import numpy as np
import seaborn as sns
import matplotlib as plt

In [12]:
DB_CONFIG = {
    "user": "postgres",
    "password": "admin",
    "host": "localhost",
    "port": "5432"
}

DB_name = "metals_db"

conn = psycopg2.connect(
    dbname="metals_db",
    user=DB_CONFIG["user"],
    password=DB_CONFIG["password"],
    host=DB_CONFIG["host"],
    port=DB_CONFIG["port"]
)

In [13]:
query = """
SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'public'
"""

tables_df = pd.read_sql(query, conn)

print(tables_df)

           table_name
0            raw_data
1          gdelt_data
2        vix_oil_data
3        cleaned_data
4       reserves_gold
5  Macroeconomic_data
6            dim_date


C:\Users\ibenl\AppData\Local\Temp\ipykernel_4132\479270233.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tables_df = pd.read_sql(query, conn)


In [14]:
data = {}

for table in tables_df['table_name']:
    try:
        print(f"Loading table: {table}")
        
        # IMPORTANT : ajouter les guillemets
        df = pd.read_sql(f'SELECT * FROM public."{table}"', conn)
        
        data[table] = df
        
        print(f"{table} loaded with shape {df.shape}")
        
    except Exception as e:
        print(f"Error loading {table}: {e}")


C:\Users\ibenl\AppData\Local\Temp\ipykernel_4132\1562415433.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f'SELECT * FROM public."{table}"', conn)


Loading table: raw_data
raw_data loaded with shape (58272, 10)
Loading table: gdelt_data
gdelt_data loaded with shape (1825411, 7)
Loading table: vix_oil_data
vix_oil_data loaded with shape (2599, 3)
Loading table: cleaned_data
cleaned_data loaded with shape (58272, 11)
Loading table: reserves_gold
reserves_gold loaded with shape (105, 4)
Loading table: Macroeconomic_data
Macroeconomic_data loaded with shape (12166, 7)
Loading table: dim_date
dim_date loaded with shape (3776, 5)


In [15]:
macro_df = data.get("Macroeconomic_data")
prices_df = data.get("cleaned_data")
geopo_df = data.get("gdelt_data")
gold_reser_df = data.get("reserves_gold")
vix_oil_df = data.get("vix_oil_data")

In [16]:
DATE_CANDIDATES = [
    "date",
    "Date",
    "datetime",
    "timestamp",
    "time"
]

In [18]:
def detect_date_column(df):
    """
    Détecte automatiquement une colonne date.
    """
    for col in df.columns:
        if col in DATE_CANDIDATES:
            return col

    for col in df.columns:
        if "date" in col.lower():
            return col

    return None


In [19]:
def detect_frequency(date_series):
    """
    Détection fréquence temporelle.
    """
    try:
        s = pd.to_datetime(date_series).sort_values()

        diffs = s.diff().dropna()

        if len(diffs) == 0:
            return "UNKNOWN"

        median_days = diffs.dt.days.median()

        if median_days <= 1:
            return "DAILY"

        elif median_days <= 7:
            return "WEEKLY"

        elif median_days <= 31:
            return "MONTHLY"

        elif median_days <= 92:
            return "QUARTERLY"

        elif median_days <= 366:
            return "YEARLY"

        else:
            return "IRREGULAR"

    except:
        return "UNKNOWN"
    

In [20]:
def temporal_gaps(date_series):
    """
    Détection des trous temporels.
    """

    try:

        s = (
            pd.to_datetime(date_series, errors="coerce")
            .dropna()
            .sort_values()
            .drop_duplicates()
        )

        if len(s) < 2:
            return {
                "nb_gaps": 0,
                "max_gap_days": 0
            }

        diffs = s.diff().dropna()

        median_gap = diffs.median()

        abnormal = diffs[diffs > median_gap * 2]

        return {
            "nb_gaps": len(abnormal),
            "max_gap_days": abnormal.max().days if len(abnormal) > 0 else 0
        }

    except:
        return {
            "nb_gaps": None,
            "max_gap_days": None
        }

In [21]:

def key_uniqueness(df):
    """
    Vérifie les colonnes potentiellement uniques.
    """

    uniqueness = {}

    for col in df.columns:

        uniqueness_ratio = (
            df[col].nunique(dropna=False) / len(df)
        )

        uniqueness[col] = round(uniqueness_ratio, 4)

    return uniqueness

In [22]:

def plot_missing_values(df, dataset_name):
    """
    Heatmap des valeurs manquantes.
    """

    plt.figure(figsize=(14, 5))

    sns.heatmap(
        df.isnull(),
        cbar=False,
        yticklabels=False
    )

    plt.title(f"Missing Values Heatmap — {dataset_name}")

    plt.show()

In [23]:
def plot_temporal_distribution(df, date_col, dataset_name):
    """
    Visualisation couverture temporelle.
    """

    tmp = (
        pd.to_datetime(df[date_col], errors="coerce")
        .dropna()
        .sort_values()
    )

    if len(tmp) == 0:
        return

    plt.figure(figsize=(14, 4))

    plt.plot(tmp.reset_index(drop=True))

    plt.title(f"Temporal Coverage — {dataset_name}")

    plt.xlabel("Index")
    plt.ylabel("Date")
    plt.show()

In [24]:
for name, df in datasets.items():

    display(Markdown(f"# 📊 DATASET : `{name}`"))

    print("=" * 100)

    # =====================================================
    # 1. STRUCTURE GLOBALE
    # =====================================================

    display(Markdown("## 1️⃣ Structure globale"))

    structure_df = pd.DataFrame({
        "Metric": [
            "Rows",
            "Columns",
            "Memory Usage (MB)"
        ],
        "Value": [
            df.shape[0],
            df.shape[1],
            round(df.memory_usage(deep=True).sum() / 1024**2, 2)
        ]
    })

    display(structure_df)

NameError: name 'datasets' is not defined